In [20]:
import numpy as np
import pandas as pd
import ast
import re

In [24]:
df_men_review = pd.read_csv('./data/xexymix_raw_review.csv')

In [26]:
df_men_review['customer_properties']

0          [{'name': '키', 'value': '158cm'}, {'name': '몸무...
1          [{'name': '키', 'value': '168cm'}, {'name': '몸무...
2          [{'name': '키', 'value': '179cm'}, {'name': '몸무...
3          [{'name': '키', 'value': '170cm'}, {'name': '몸무...
4          [{'name': '키', 'value': '153cm'}, {'name': '몸무...
                                 ...                        
2364097                                                   []
2364098    [{'name': '키', 'value': '161cm'}, {'name': '몸무...
2364099                                                   []
2364100                                                   []
2364101                                                   []
Name: customer_properties, Length: 2364102, dtype: str

In [29]:
df_men_review['customer_properties_values']

0           [158cm, 50~54kg, M]
1           [168cm, 55~59kg, M]
2          [179cm, 70~74kg, XL]
3           [170cm, 60~64kg, L]
4           [153cm, 45~49kg, S]
                   ...         
2364097                      []
2364098     [161cm, 50~54kg, S]
2364099                      []
2364100                      []
2364101                      []
Name: customer_properties_values, Length: 2364102, dtype: object

# 상품 정보

In [50]:
# df_men_info = pd.read_json('xexymix_men_info_260420.json')
# df_men_info.info()

## 1. category = gender로 변경

In [51]:
df_men_info = df_men_info.rename(columns={"category": "gender"})

## 2. collect_date = 2026.04.20

In [52]:
df_men_info["collect_date"] = pd.to_datetime("2026-04-20")
df_men_info["collect_date"] = df_men_info["collect_date"].dt.date

In [54]:
print(df_men_info.columns.tolist())

['id', 'score', 'store_name', 'brand_user_id', 'user_display_name', 'media_count', 'product_code', 'product_image_source_url', 'product_image_url', 'product_name', 'product_meta_reviews_count', 'product_meta_score', 'social_media_type', 'product_url', 'created_at', 'editable', 'deletable', 'tags', 'likes_count', 'plus_likes_count', 'reported', 'my_review', 'author_grade', 'like_score', 'comments_count', 'review_vendor', 'external_platform_type', 'review_source', 'review_source_has_video', 'filtered_message', 'content_type', 'review_source_url', 'verified_buyer_badge_image_url', 'ai_summary', 'brand_user_grade', 'videos', 'images', 'product_options', 'customer_properties', 'brand_author_type', 'gender', 'brand', 'customer_properties_parsed', 'customer_properties_values', 'user_height', 'user_weight', 'user_size', 'collect_date']


## 3. original_price,discount_rate,discount_price,coupon_price 형변환

In [ ]:
cols = ["original_price", "discount_price", "coupon_price",'review_count']

df_men_info[cols] = (
    df_men_info[cols]
    .astype(str)
    .replace(r"[,\s%]", "", regex=True)
    .replace("", np.nan)
)

df_men_info[cols] = df_men_info[cols].apply(pd.to_numeric, errors="coerce")


df_men_info[cols] = df_men_info[cols].astype("Int64")

KeyError: "None of [Index(['original_price', 'discount_price', 'coupon_price', 'review_count'], dtype='str')] are in the [columns]"

In [ ]:
df_men_info["discount_rate"] = (
    df_men_info["discount_rate"]
    .astype(str)
    .str.replace("%", "", regex=False)  # % 제거
    .str.strip()                        # 공백 제거
)

df_men_info["discount_rate"] = pd.to_numeric(df_men_info["discount_rate"], errors="coerce")

## 4. category 추가하기

In [ ]:
import re
from kiwipiepy import Kiwi

In [ ]:
df_men_info['category'] = df_men_info['product_name']

In [ ]:
import re
from kiwipiepy import Kiwi

kiwi = Kiwi()

# =========================
# 1. 전처리
# =========================
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^가-힣a-z0-9 ]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# =========================
# 2. 형태소 분석 (Kiwi)
# =========================
def extract_nouns(text):
    tokens = kiwi.tokenize(text)
    return [t.form for t in tokens if t.tag.startswith('N')]


# =========================
# 3. 불용어
# =========================
STOPWORDS = {
    "맨즈","맨","즈",
    "매직","올웨이즈","퍼펙트",
    "아이스","쿨","쿨브리즈","브리즈","텐션","라이트",
    "로고","포인트","라인",
    "텐","션","테이","퍼","드","카","부","파이","핑","웃",
    "데이","백로","스트링",
    "빅","시어서커"
}

FORCE_KEYWORDS = {
    "바람막이", "윈드브레이커", "하프집업",
    "후드집업", "카고쇼츠", "스웨트셔츠",
    "맨즈레깅스", "세트", "셋업", "롱슬리브",
    "숏슬리브","래쉬가드", "아노락","레깅스"
}


# =========================
# 4. 카테고리 사전
# =========================
CATEGORY_DICT = {

    "상의": [
        "티셔츠", "반팔", "롱슬리브", "숏슬리브", "슬리브리스",
        "셔츠", "니트", "맨투맨", "스웨트셔츠", "후드", "후디",
        "후드티", "피케", "폴로", "저지", "탑", "브이넥",
        "라운드", "모크넥", "하프넥", "스웨트", "플리스", "래쉬가드"
    ],

    "하의": [
        "팬츠", "슬랙스", "조거", "조깅스",
        "쇼츠", "카고쇼츠", "버뮤다", "5부",
        "레깅스", "데님", "카고"
    ],

    "아우터": [
        "자켓", "코트", "패딩", "점퍼",
        "바람막이", "윈드브레이커", "브레이커",
        "아노락", "셔켓"
    ],

    "셋업": [
        "세트", "셋업"
    ]
}

PRIORITY = ["셋업", "아우터", "상의", "하의"]

# =========================
# 5. 키워드 보정
# =========================
def recover_keywords(text, nouns):
    result = set(nouns)

    for kw in FORCE_KEYWORDS:
        if kw in text:
            result.add(kw)

    return list(result)


# =========================
# 6. STOPWORDS + 정제
# =========================
def clean_nouns(nouns):
    return [
        n for n in nouns
        if n not in STOPWORDS and len(n) > 1
    ]


# =========================
# 7. 카테고리 매핑
# =========================
def map_category(nouns):
    matched = []

    for noun in nouns:
        for category, keywords in CATEGORY_DICT.items():
            if noun in keywords:
                matched.append(category)

    if not matched:
        return ["기타"]

    return matched


# =========================
# 8. 최종 카테고리 결정
# =========================
def resolve_category(categories):
    for p in PRIORITY:
        if p in categories:
            return p
    return "기타"


# =========================
# 9. 전체 파이프라인
# =========================
def classify_category(name):
    text = preprocess(name)

    nouns = extract_nouns(text)
    nouns = recover_keywords(text, nouns)
    nouns = clean_nouns(nouns)

    categories = map_category(nouns)

    return resolve_category(categories)


# =========================
# 10. DataFrame 적용
# =========================
df_men_info['category'] = df_men_info['product_name'].apply(classify_category)

In [ ]:
df_men_info[df_men_info['discount_price'].isnull()]

,brand,gender,page,product_id,product_name,original_price,discount_rate,discount_price,coupon_price,color_count,raw_colors,rating,review_count,collect_date,category
76,xexymix,men,1,2077170,[덱스PICK] NX 맨즈 로고밴딩 투인원 쇼츠,<NA>,NaN,<NA>,60720,4,"[#000000, #D5D4D5, #474D5B, #695B55]",5.0,11,2026-04-20,하의
383,xexymix,men,2,2072187,[덱스PICK] 웜 소프트 맨즈 핀턱 와이드팬츠,<NA>,NaN,<NA>,34320,4,"[#766761, #86836C, #BDBEBE, #000000]",4.9,588,2026-04-20,하의
384,xexymix,men,2,2068434,에센셜 시그니처 기모 스웨트셔츠,<NA>,NaN,<NA>,54560,4,"[#000000, #D3D3D2, #746C57, #D1CCB3]",4.9,795,2026-04-20,상의
385,xexymix,men,2,2073213,RX 맨즈 기모 포켓 레깅스,<NA>,NaN,<NA>,51920,2,"[#46494E, #000000]",4.9,395,2026-04-20,하의


## 5. 스키마 칼럼만

In [ ]:
final_men_info_df = df_men_info[[
    "collect_date",
    "brand",
    "product_id",
    "product_name",
    "category",
    "gender",
    "original_price",
    "discount_price",
    "color_count",
    "raw_colors",
    "review_count"
]]

In [ ]:
final_men_info_df.to_csv(
    "xexymix_men_info.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
final_men_info_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 386 entries, 0 to 385
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   collect_date    386 non-null    object
 1   brand           386 non-null    object
 2   product_id      386 non-null    int64 
 3   product_name    386 non-null    object
 4   category        386 non-null    object
 5   gender          386 non-null    object
 6   original_price  186 non-null    Int64 
 7   discount_price  382 non-null    Int64 
 8   color_count     386 non-null    int64 
 9   raw_colors      386 non-null    object
 10  review_count    372 non-null    Int64 
dtypes: Int64(3), int64(2), object(6)
memory usage: 34.4+ KB


# 상품 리뷰

In [ ]:
# import json
# import pandas as pd

# data = []

# with open("xexymix_man_review_260420.json", "r", encoding="utf-8") as f:
#     for line in f:
#         line = line.strip()

#         if not line:
#             continue

#         try:
#             data.append(json.loads(line))
#         except:
#             pass
        
# df_men_review = pd.DataFrame(data)
# df_men_review.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 343901 entries, 0 to 343900
Data columns (total 42 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              343901 non-null  int64  
 1   score                           343901 non-null  int64  
 2   store_name                      0 non-null       object 
 3   brand_user_id                   309733 non-null  float64
 4   user_display_name               343901 non-null  object 
 5   media_count                     343901 non-null  int64  
 6   product_code                    343901 non-null  int64  
 7   product_image_source_url        343901 non-null  object 
 8   product_image_url               343901 non-null  object 
 9   product_name                    343901 non-null  object 
 10  product_meta_reviews_count      343901 non-null  int64  
 11  product_meta_score              343901 non-null  float64
 12  social_media_typ

In [39]:
import pandas as pd 

df_men_review = pd.read_csv('./data/xexymix_raw_review.csv')

## 1. 젝시믹스 분석 기간인 2024년 이후로

In [ ]:
# #2024년 이후
# df_men_review['year'] = pd.to_datetime(df_men_review['created_at'],format = 'mixed', utc=True).dt.year
# print(df_men_review['year'].value_counts().sort_index())

# df_men_review = df_men_review[df_men_review['year']>=2024]
# #원본 : 343901 , 2024년 이후 : 232,476 
# print(df_men_review['year'].value_counts().sort_index())

year
2020       585
2021      8956
2022     28887
2023     72447
2024    107967
2025    105978
2026     19081
Name: count, dtype: int64
year
2024    107967
2025    105978
2026     19081
Name: count, dtype: int64


## 2. collect_date = 2026.04.20

In [40]:
df_men_review["collect_date"] = pd.to_datetime("2026-04-20")
df_men_review["collect_date"] = df_men_review["collect_date"].dt.date

## 3. created_at > post_date

In [41]:
# 날짜 형식으로 변환
df_men_review["created_at"] = pd.to_datetime(
    df_men_review["created_at"], 
    format="ISO8601"
)

# 시간 정보를 제외한 날짜만 추출
df_men_review["created_at"] = df_men_review["created_at"].dt.date

# 컬럼명 변경
df_men_review = df_men_review.rename(columns={"created_at": "post_date"})

In [42]:
df_men_review['post_date']

0          2026-03-09
1          2026-01-26
2          2025-12-14
3          2025-07-22
4          2025-10-10
              ...    
2364097    2021-05-20
2364098    2018-08-14
2364099    2020-06-30
2364100    2020-06-30
2364101    2020-06-30
Name: post_date, Length: 2364102, dtype: object

In [7]:
# # 날짜 형식으로 변환 (이 작업이 반드시 먼저 필요합니다)
# df_men_review["created_at"] = pd.to_datetime(df_men_review["created_at"])

# # 시간 정보를 제외한 날짜만 추출
# df_men_review["created_at"] = df_men_review["created_at"].dt.date

# # 컬럼명 변경
# df_men_review = df_men_review.rename(columns={"created_at": "post_date"})

## 4. platform  = 자사몰 생성

In [43]:
df_men_review['platform'] = '자사몰'

## 5. filtered_message > content

In [44]:
df_men_review = df_men_review.rename(columns={"filtered_message": "content"})

## 6. product_url > source_url

In [45]:
df_men_review = df_men_review.rename(columns={"product_url": "source_url"})

## 7. score > rating

In [46]:
df_men_review = df_men_review.rename(columns={"score": "rating"})

## 8.likes_count(plus_likes_count) > helpful_count

In [47]:
df_men_review = df_men_review.rename(columns={"likes_count": "helpful_count"})

## 9.product_options > purchased_option

In [48]:
df_men_review = df_men_review.rename(columns={"product_options": "purchased_option"})

## 10.product_options 기반 has_image 칼럼 생성

In [49]:
df_men_review["has_image"] = df_men_review["images"].apply(bool).astype(int)

In [50]:
df_men_review.info()

<class 'pandas.DataFrame'>
RangeIndex: 2364102 entries, 0 to 2364101
Data columns (total 45 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   id                              int64  
 1   rating                          int64  
 2   store_name                      float64
 3   brand_user_id                   float64
 4   user_display_name               str    
 5   media_count                     int64  
 6   product_code                    int64  
 7   product_image_source_url        str    
 8   product_image_url               str    
 9   product_name                    str    
 10  product_meta_reviews_count      int64  
 11  product_meta_score              float64
 12  social_media_type               int64  
 13  source_url                      str    
 14  post_date                       object 
 15  editable                        bool   
 16  deletable                       bool   
 17  tags                            str   

## 스키마 칼럼만

In [51]:
df_men_review["search_keyword"] = None

In [52]:
df_men_review

,id,rating,store_name,brand_user_id,user_display_name,media_count,product_code,product_image_source_url,product_image_url,product_name,...,images,purchased_option,customer_properties,brand_author_type,category,brand,collect_date,platform,has_image,search_keyword
0,5032253,5,NaN,52794605.0,박**,1,2070608,//www.xexymix.com/shopimages/xexymix/035002000...,https://assets4.cre.ma/p/xexymix-com/products/...,UV 쉴드 에어핏 페이스 마스크,...,[{'url': 'https://assets4.cre.ma/p/xexymix-com...,"[{'name': '색상', 'value': '화이트'}, {'name': '사이즈...","[{'name': '키', 'value': '158cm'}, {'name': '몸무...",NaN,men,xexymix,2026-04-20,자사몰,1,None
1,4998316,5,NaN,52681467.0,김**,1,2070608,//www.xexymix.com/shopimages/xexymix/035002000...,https://assets4.cre.ma/p/xexymix-com/products/...,UV 쉴드 에어핏 페이스 마스크,...,[{'url': 'https://assets4.cre.ma/p/xexymix-com...,"[{'name': '색상', 'value': '블랙'}, {'name': '사이즈'...","[{'name': '키', 'value': '168cm'}, {'name': '몸무...",NaN,men,xexymix,2026-04-20,자사몰,1,None
2,4936161,5,NaN,52688236.0,임**,1,2070608,//www.xexymix.com/shopimages/xexymix/035002000...,https://assets4.cre.ma/p/xexymix-com/products/...,UV 쉴드 에어핏 페이스 마스크,...,[],"[{'name': '색상', 'value': '블랙'}, {'name': '사이즈'...","[{'name': '키', 'value': '179cm'}, {'name': '몸무...",NaN,men,xexymix,2026-04-20,자사몰,1,None
3,4727267,5,NaN,25233397.0,김**,1,2070608,//www.xexymix.com/shopimages/xexymix/035002000...,https://assets4.cre.ma/p/xexymix-com/products/...,UV 쉴드 에어핏 페이스 마스크,...,[{'url': 'https://assets4.cre.ma/p/xexymix-com...,"[{'name': '색상', 'value': '화이트'}, {'name': '사이즈...","[{'name': '키', 'value': '170cm'}, {'name': '몸무...",NaN,men,xexymix,2026-04-20,자사몰,1,None
4,4821424,5,NaN,30053251.0,박**,1,2070608,//www.xexymix.com/shopimages/xexymix/035002000...,https://assets4.cre.ma/p/xexymix-com/products/...,UV 쉴드 에어핏 페이스 마스크,...,[{'url': 'https://assets4.cre.ma/p/xexymix-com...,"[{'name': '색상', 'value': '화이트'}, {'name': '사이즈...","[{'name': '키', 'value': '153cm'}, {'name': '몸무...",NaN,men,xexymix,2026-04-20,자사몰,1,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2364097,1400261,5,NaN,29629573.0,*,1,2058783,//www.xexymix.com/shopimages/xexymix/033001000...,https://assets4.cre.ma/p/xexymix-com/products/...,스파키 숏슬리브 백아이보리,...,[{'url': 'https://assets4.cre.ma/p/xexymix-com...,"[{'name': 'SIZE', 'value': 'M(55반~66)'}]",[],NaN,women,xexymix,2026-04-20,자사몰,1,None
2364098,75158,3,NaN,195056.0,장**,1,2058783,//www.xexymix.com/shopimages/xexymix/033001000...,https://assets4.cre.ma/p/xexymix-com/products/...,"XA068 레드,캔디핑크,블랙,화이트,딤그레이",...,[{'url': 'https://assets4.cre.ma/p/xexymix-com...,"[{'name': '컬러', 'value': '화이트-교환및반품불가동의'}]","[{'name': '키', 'value': '161cm'}, {'name': '몸무...",NaN,women,xexymix,2026-04-20,자사몰,1,None
2364099,763803,1,NaN,934864.0,유**,1,2058783,//www.xexymix.com/shopimages/xexymix/033001000...,https://assets4.cre.ma/p/xexymix-com/products/...,스파키 숏슬리브 밀키그레이,...,[{'url': 'https://assets4.cre.ma/p/xexymix-com...,"[{'name': 'SIZE', 'value': 'S(44~55)'}]",[],NaN,women,xexymix,2026-04-20,자사몰,1,None
2364100,763800,1,NaN,934864.0,유**,1,2058783,//www.xexymix.com/shopimages/xexymix/033001000...,https://assets4.cre.ma/p/xexymix-com/products/...,스파키 숏슬리브 블랑실버,...,[{'url': 'https://assets4.cre.ma/p/xexymix-com...,"[{'name': 'SIZE', 'value': 'S(44~55)'}]",[],NaN,women,xexymix,2026-04-20,자사몰,1,None


In [38]:
df_men_review.columns

Index(['id', 'score', 'store_name', 'brand_user_id', 'user_display_name',
       'media_count', 'product_code', 'product_image_source_url',
       'product_image_url', 'product_name', 'product_meta_reviews_count',
       'product_meta_score', 'social_media_type', 'product_url', 'created_at',
       'editable', 'deletable', 'tags', 'likes_count', 'plus_likes_count',
       'reported', 'my_review', 'author_grade', 'like_score', 'comments_count',
       'review_vendor', 'external_platform_type', 'review_source',
       'review_source_has_video', 'filtered_message', 'content_type',
       'review_source_url', 'verified_buyer_badge_image_url', 'ai_summary',
       'brand_user_grade', 'videos', 'images', 'product_options',
       'customer_properties', 'brand_author_type', 'category', 'brand',
       'customer_properties_parsed', 'customer_properties_values',
       'user_height', 'user_weight', 'user_size'],
      dtype='str')

- 키, 몸무게 추출

In [57]:
def parse_str_to_list(x):
    if not x or x == '[]':
        return []
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return []

df_men_review['customer_properties_parsed'] = df_men_review['customer_properties'].apply(parse_str_to_list)

In [58]:
df_men_review['customer_properties_values'] = df_men_review['customer_properties_parsed'].apply(
    lambda lst: [d.get('value') for d in lst] if lst else []
)

In [59]:
def extract_number(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r'\d+\.?\d*', str(value))
    return float(match.group()) if match else np.nan

df_men_review['user_height'] = df_men_review['customer_properties_values'].apply(lambda x: x[0] if len(x) > 0 else np.nan)
df_men_review['user_weight'] = df_men_review['customer_properties_values'].apply(lambda x: extract_number(x[1]) if len(x) > 1 else np.nan)
df_men_review['user_size'] = df_men_review['customer_properties_values'].apply(lambda x: x[2] if len(x) > 2 else np.nan)

df_men_review['user_height'] = df_men_review['user_height'].str.replace('cm', '', regex=False)

In [60]:
final_men_review_df = df_men_review[[
    "collect_date",
    "post_date",
    "platform",
    "brand",
    "search_keyword",
    "content",
    "source_url",
    "rating",
    "helpful_count",
    "purchased_option",
    "has_image",
    "user_height",
    "user_weight",
    "user_size"
]]

In [64]:
final_men_review_df.to_csv(
    "xexymix_raw_review_final.csv",
    index=False,
    encoding="utf-8-sig"
)

In [62]:
final_men_review_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2364102 entries, 0 to 2364101
Data columns (total 14 columns):
 #   Column            Dtype  
---  ------            -----  
 0   collect_date      object 
 1   post_date         object 
 2   platform          str    
 3   brand             str    
 4   search_keyword    object 
 5   content           str    
 6   source_url        str    
 7   rating            int64  
 8   helpful_count     int64  
 9   purchased_option  str    
 10  has_image         int64  
 11  user_height       str    
 12  user_weight       float64
 13  user_size         str    
dtypes: float64(1), int64(3), object(3), str(7)
memory usage: 252.5+ MB


In [63]:
df_men_review['product_image_url'].isnull().sum()

np.int64(0)

In [ ]:
(set(df_men_info["product_id"]) - set(review_count["product_code"]))

{2073893,
 2073943,
 2075462,
 2075464,
 2075465,
 2075466,
 2075561,
 2075599,
 2075600,
 2077107,
 2077109,
 2077224,
 2077283,
 2077360,
 2077396}

In [ ]:
df_men_info[df_men_info["product_id"]==2077396]

,brand,gender,page,product_id,product_name,original_price,discount_rate,discount_price,coupon_price,collect_date,product_code_x,product_meta_reviews_count_x,product_image_url_x,product_code_y,product_meta_reviews_count_y,product_image_url_y
27,xexymix,men,1,2077396,소프트 릴렉스 맨즈 슬리브리스,<NA>,NaN,34000,30600,2026-04-18,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
unmatched = set(df_men_info["product_id"]) - set(review_count["product_code"])
unmatched
df_men_review[df_men_review["product_code"].isin(unmatched)]

,id,rating,store_name,brand_user_id,user_display_name,media_count,product_code,product_image_source_url,product_image_url,product_name,...,videos,images,purchased_option,customer_properties,brand_author_type,category,brand,year,collect_date,platform


In [ ]:
df_men_info["product_id"].dtype
df_men_review["product_code"].dtype

dtype('int64')

In [ ]:
review_count

,product_code,product_meta_reviews_count,product_image_url
0,2060391,13977,https://assets4.cre.ma/p/xexymix-com/products/...
1,2061003,8179,https://assets4.cre.ma/p/xexymix-com/products/...
2,2061203,15421,https://assets4.cre.ma/p/xexymix-com/products/...
3,2061309,11239,https://assets4.cre.ma/p/xexymix-com/products/...
4,2061596,1759,https://assets4.cre.ma/p/xexymix-com/products/...
...,...,...,...
367,2077506,4,https://assets4.cre.ma/p/xexymix-com/products/...
368,2077677,3,https://assets4.cre.ma/p/xexymix-com/products/...
369,2077678,4,https://assets4.cre.ma/p/xexymix-com/products/...
370,2077689,2,https://assets4.cre.ma/p/xexymix-com/products/...


In [ ]:
df_men_info = df_men_info.merge(
    review_count,
    left_on="product_id",
    right_on="product_code",
    how="left"
)

In [ ]:
df_men_info["product_code"].isnull().sum()

np.int64(15)

In [ ]:
df_men_info

,brand,gender,page,product_id,product_name,original_price,discount_rate,discount_price,coupon_price,collect_date,product_code_x,product_meta_reviews_count_x,product_image_url_x,product_code_y,product_meta_reviews_count_y,product_image_url_y
0,xexymix,men,1,2066959,[3PACK] 맨즈 아이스페더 숏슬리브,117000,61.0,44800,40320,2026-04-18,2066959.0,11152.0,https://assets4.cre.ma/p/xexymix-com/products/...,2066959.0,11152.0,https://assets4.cre.ma/p/xexymix-com/products/...
1,xexymix,men,1,2077455,[2PACK] 매직밴딩 맨즈 슬랙스 9.1&9.6,158000,21.0,124800,112320,2026-04-18,2077455.0,5670.0,https://assets4.cre.ma/p/xexymix-com/products/...,2077455.0,5670.0,https://assets4.cre.ma/p/xexymix-com/products/...
2,xexymix,men,1,2077442,[2PACK] 쿨브리즈 오버핏 숏슬리브,73000,12.0,63800,57420,2026-04-18,2077442.0,741.0,https://assets4.cre.ma/p/xexymix-com/products/...,2077442.0,741.0,https://assets4.cre.ma/p/xexymix-com/products/...
3,xexymix,men,1,2072026,올웨이즈 퍼펙트텐션 맨즈 쇼츠 5&5.5부,<NA>,NaN,49000,44100,2026-04-18,2072026.0,1911.0,https://assets4.cre.ma/p/xexymix-com/products/...,2072026.0,1911.0,https://assets4.cre.ma/p/xexymix-com/products/...
4,xexymix,men,1,2075462,백로고 반팔 래쉬가드,<NA>,NaN,46000,41400,2026-04-18,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
382,xexymix,men,2,2068522,RX 맨즈 프레시 웜 풀커버 롱슬리브,74000,20.0,59000,53100,2026-04-18,2068522.0,152.0,https://assets4.cre.ma/p/xexymix-com/products/...,2068522.0,152.0,https://assets4.cre.ma/p/xexymix-com/products/...
383,xexymix,men,2,2068435,에센셜 시그니처 기모 스웨트후디,<NA>,NaN,79000,71100,2026-04-18,2068435.0,2551.0,https://assets4.cre.ma/p/xexymix-com/products/...,2068435.0,2551.0,https://assets4.cre.ma/p/xexymix-com/products/...
384,xexymix,men,2,2072187,[덱스PICK] 웜 소프트 맨즈 핀턱 와이드팬츠,59000,33.0,39000,35100,2026-04-18,2072187.0,588.0,https://assets4.cre.ma/p/xexymix-com/products/...,2072187.0,588.0,https://assets4.cre.ma/p/xexymix-com/products/...
385,xexymix,men,2,2068434,에센셜 시그니처 기모 스웨트셔츠,<NA>,NaN,<NA>,55800,2026-04-18,2068434.0,2551.0,https://assets4.cre.ma/p/xexymix-com/products/...,2068434.0,2551.0,https://assets4.cre.ma/p/xexymix-com/products/...
